## Feature Engineering

In [42]:
import pandas as pd
import numpy as np
import json
import joblib
import os
import warnings
from sklearn.preprocessing import OrdinalEncoder
warnings.filterwarnings('ignore')
print(f'imports successful')

imports successful


In [2]:
# Load the feature whitelist data preparation and EDA
with open('../data/processed/consistent_features.json') as f:
    cfg = json.load(f)
CONSISTENT_FEATURES = cfg['consistent_features']
MODEL_EXCLUDED = cfg['model_excluded']
LEAKAGE_COLUMNS = cfg['leakage_columns']

df = pd.read_parquet('../data/processed/loans_dev_100k.parquet')
df['issue_d_parsed'] = pd.to_datetime(df['issue_d'], format='%b-%Y', errors='coerce')
df['issue_year']   = df['issue_d_parsed'].dt.year
df['issue_quarter'] = df['issue_d_parsed'].dt.quarter
print(f'Loaded dev sample: {df.shape}')

Loaded dev sample: (100000, 155)


## Feature Engineering: Borrower Affordability, Debt Burden and Credit Quality

In [9]:
# All features come from origination-time data only
# No macro data at the row level 
# Uniform binning: every pd.cut uses right=False (left-inclusive [low, high))


# Affordability ratio: requested loan size relative to income; clip income to avoid divide-by-zero. Higher = more stretched borrower.
df['loan_to_income']   = df['loan_amnt'] / df['annual_inc'].clip(lower=1)

# Debt-to-income bucketed into risk tiers. 
df['dti_band']  = pd.cut(df['dti'], bins=[0,15,25,35,np.inf],
                                  labels=['Low','Med','High','VeryHigh'], right=False)

# FICO midpoint bucketed into credit-quality tiers using Royal Bank of Canada (RBC) labels.
# [300,640) Poor,[640,720) Fair, [720,760) Good, [760,800) VeryGood, [800,900) Excellent.  
df['fico_band'] = pd.cut(
    (df['fico_range_low'] + df['fico_range_high']) / 2,
    bins=[300,640,720,760,800,900],
    labels=['Poor','Fair','Good','VeryGood','Excellent'],
    right=False
)

# Revolving credit utilisation bucketed. MaxedOut = at/over the credit limit (>=100%).
# [0,30) Low, [30,60) Med, [60,100) High, [100, inf) MaxedOut.
df['credit_util_band']  = pd.cut(df['revol_util'],
                                  bins=[0,30,60,100,np.inf],
                                  labels=['Low','Med','High','MaxedOut'], right=False)
print('Features engineered.')

Features engineered.


## Feature Engineering: Employment, Credit Behaviour and Vintage Features

In [10]:
# Parse numeric years out of the "X years" string; missing/unknown -> 0.
df['employment_years'] = (
    df['emp_length']
      .replace({'< 1 year': '0.5 years'})
      .str.extract(r'(\d+\.?\d*)')
      .astype(float)
      .fillna(0)
)
# Binary flag for thin job tenure (<1 yr), a known instability marker.
df['short_tenure_flag'] = (df['employment_years'] < 1).astype(int)

# Origination year — structural time dimension (vintage), NOT a macro covariate.
df['vintage_year']      = df['issue_year']

# Origination quarter — finer-grained vintage slice for cohort effects.
df['vintage_quarter']   = df['issue_quarter']

# Number of open credit lines bucketed; count is unbounded, so no upper clip.
# Negatives -> NaN (invalid count surfaces as missing rather than Few).
# Left-inclusive: [0,5) Few, [5,10) Some, [10,20) Many, [20, inf) VeryMany.
df['open_acc_band']     = pd.cut(df['open_acc'],
                                  bins=[0,5,10,20,np.inf],
                                  labels=['Few','Some','Many','VeryMany'], right=False)

# Binary flag: any delinquency in the last 2 years.
df['delinq_flag']       = (df['delinq_2yrs'] > 0).astype(int)

# Binary flag: any derogatory public record on file.
df['pub_rec_flag']      = (df['pub_rec'] > 0).astype(int)

# Remap loan purpose to a coarse risk tier based on observed EDA default rates;
# unseen purposes default to Med.
purpose_map = {
    'car':'Low','home_improvement':'Low','major_purchase':'Low','vacation':'Low',
    'credit_card':'Med','debt_consolidation':'Med','moving':'Med','wedding':'Med','house':'Med',
    'small_business':'High','medical':'High','educational':'High','other':'High',
    'renewable_energy':'High'
}
# Apply the purpose->tier mapping, defaulting anything unmapped to 'Med'.
df['purpose_risk_tier'] = df['purpose'].map(purpose_map).fillna('Med')

print('Features engineered.')

Features engineered.


## Build features from CONSISTENT_FEATURES only

In [35]:
# All features — raw passthroughs, engineered bands, and vintage cohort identifiers.
# vintage_year and vintage_quarter approved via ANALYST_OVERRIDES in data_preparation.
PASSTHROUGH = {'loan_amnt', 'annual_inc', 'home_ownership'}

FEATURE_LIST = [
    'loan_to_income', 'dti_band', 'fico_band', 'credit_util_band',
    'employment_years', 'short_tenure_flag', 'vintage_year', 'vintage_quarter',
    'open_acc_band', 'delinq_flag', 'pub_rec_flag', 'purpose_risk_tier',
    'loan_amnt', 'annual_inc', 'home_ownership',
]

# Retain only features that exist in df and whose passthroughs are approved
FEATURE_LIST = [
    f for f in FEATURE_LIST
    if f in df.columns
    and (f not in PASSTHROUGH or f in CONSISTENT_FEATURES)
]

print('CONSISTENT_FEATURES audit PASSED')
print()
print(f'Final feature count: {len(FEATURE_LIST)}')
print()
print(FEATURE_LIST)

CONSISTENT_FEATURES audit PASSED

Final feature count: 15

['loan_to_income', 'dti_band', 'fico_band', 'credit_util_band', 'employment_years', 'short_tenure_flag', 'vintage_year', 'vintage_quarter', 'open_acc_band', 'delinq_flag', 'pub_rec_flag', 'purpose_risk_tier', 'loan_amnt', 'annual_inc', 'home_ownership']


## Vintage Cohort Train, Validation and Test Split

In [19]:
# Our data has a vintage_year column — loans originated in different years. 
# Temporal split by vintage_year — train on earlier cohorts, validate and test on
# progressively later ones to simulate out-of-time production scoring and prevent
# vintage-level leakage from inflating evaluation metrics.

train_mask = df['vintage_year'] <= 2015
val_mask   = df['vintage_year'] == 2016
test_mask  = df['vintage_year'].isin([2017, 2018])

X_train = df.loc[train_mask, FEATURE_LIST].copy()
y_train = df.loc[train_mask, 'default_flag'].copy()
X_val   = df.loc[val_mask,   FEATURE_LIST].copy()
y_val   = df.loc[val_mask,   'default_flag'].copy()
X_test  = df.loc[test_mask,  FEATURE_LIST].copy()
y_test  = df.loc[test_mask,  'default_flag'].copy()


datasets = {
    'Train': (X_train, y_train),
    'Validation': (X_val, y_val),
    'Test': (X_test, y_test)
}

for dataset_name, (X, y) in datasets.items():
    print(
        f"{dataset_name}: "
        f"{len(X):,} rows | "
        f"Default Rate: {y.mean():.2%}"
    )

Train: 61,442 rows | Default Rate: 18.43%
Validation: 21,786 rows | Default Rate: 23.29%
Test: 16,772 rows | Default Rate: 21.29%


## Train / Validation / Test Split — Temporal Design

**Boundaries**
| Split      | Vintages   | Rows   | Default Rate |
|------------|------------|--------|--------------|
| Train      | ≤ 2015     | 61,442 | 18.43%       |
| Validation | 2016       | 21,786 | 23.29%       |
| Test       | 2017–2018  | 16,772 | 21.29%       |

**Why temporal, not random**
Loans are split by origination year rather than randomly to simulate out-of-time
production scoring and prevent vintage-level leakage from inflating evaluation metrics.

**Why these boundaries**
- Including 2015 in train preserves volume (61k rows) and keeps train as the largest slice.
- 2016 alone as validation provides ~22k rows — sufficient for reliable hyperparameter tuning.
- 2017–2018 as test resolves the maturity problem of using 2018 alone (too few rows,
  artificially low default rate due to loans not yet having enough time to default).

**The 2016 elevation**
Validation default rate (23.29%) is ~5pp above train (18.43%). This reflects
LendingClub's documented 2016 credit quality deterioration — a real data property,
not a split artefact. Expect validation metrics to look slightly harder than test;
this is intentional and correct.

**Validation vs test alignment**
Test default rate (21.29%) closely matches validation (23.29%), meaning the model
is tuned and evaluated in consistent default-rate environments.

## Encode Categorical Variables and Save Modeling Datasets

In [41]:
# Categorical columns — vintage_year and vintage_quarter excluded (int32, natural numeric order)
CAT_COLS = [
    'dti_band', 'fico_band', 'credit_util_band', 'open_acc_band',
    'purpose_risk_tier', 'home_ownership'
]

# Explicit category order ensures ordinal ranks carry real signal.
# Fit on train only — transform val and test without refitting.
encoder = OrdinalEncoder(
    categories=[
        ['Low', 'Med', 'High', 'VeryHigh'],                  # dti_band
        ['Poor', 'Fair', 'Good', 'VeryGood', 'Excellent'],   # fico_band
        ['Low', 'Med', 'High', 'VeryHigh', 'MaxedOut'],      # credit_util_band
        ['Few', 'Some', 'Many', 'VeryMany'],                  # open_acc_band
        ['Low', 'Med', 'High'],                               # purpose_risk_tier
        ['NONE', 'ANY', 'OTHER', 'RENT', 'OWN', 'MORTGAGE'], # home_ownership
    ],
    handle_unknown='use_encoded_value',
    unknown_value=-1
)

X_train_enc = X_train.copy()
X_val_enc   = X_val.copy()
X_test_enc  = X_test.copy()

X_train_enc[CAT_COLS] = encoder.fit_transform(X_train[CAT_COLS].astype(str))
X_val_enc[CAT_COLS]   = encoder.transform(X_val[CAT_COLS].astype(str))
X_test_enc[CAT_COLS]  = encoder.transform(X_test[CAT_COLS].astype(str))

# Persist encoder and splits
os.makedirs('models', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

joblib.dump(encoder, 'models/ordinal_encoder.pkl')

for name, X in [('X_train', X_train_enc), ('X_val', X_val_enc), ('X_test', X_test_enc)]:
    X.to_parquet(f'../data/processed/{name}.parquet', index=False)

for name, y in [('y_train', y_train), ('y_val', y_val), ('y_test', y_test)]:
    y.to_frame().to_parquet(f'../data/processed/{name}.parquet', index=False)

# Save feature list — used by NB09, NB10, NB11, Streamlit app
with open('models/feature_list.txt', 'w') as f:
    f.write('\n'.join(FEATURE_LIST))

print('Saved: encoder, feature_list.txt, all 6 split parquets')
print()
print('Number of Feature list:', len(FEATURE_LIST))
print()
print(f"Data types of features in FEATURE_LIST:\n{df[FEATURE_LIST].dtypes}")
print()
print('Feature list:', FEATURE_LIST)

Saved: encoder, feature_list.txt, all 6 split parquets

Number of Feature list: 15

Data types of features in FEATURE_LIST:
loan_to_income        float64
dti_band             category
fico_band            category
credit_util_band     category
employment_years      float64
short_tenure_flag       int64
vintage_year            int32
vintage_quarter         int32
open_acc_band        category
delinq_flag             int64
pub_rec_flag            int64
purpose_risk_tier      object
loan_amnt             float64
annual_inc            float64
home_ownership         object
dtype: object

Feature list: ['loan_to_income', 'dti_band', 'fico_band', 'credit_util_band', 'employment_years', 'short_tenure_flag', 'vintage_year', 'vintage_quarter', 'open_acc_band', 'delinq_flag', 'pub_rec_flag', 'purpose_risk_tier', 'loan_amnt', 'annual_inc', 'home_ownership']
